# 13 - Encode external ECG and CXR data with the Symile bridge

This notebook first creates frozen HuBERT-ECG and RAD-DINO features, then applies every trained bridge ablation. It writes `continuous_common`, final normalized `common`, and shared code indices where applicable. HuBERT and RAD-DINO are loaded sequentially to limit GPU memory use. Status arrays make failed records explicit and allow a rerun to resume.

Set `EXTERNAL_MAX_ROWS` to a small positive number for a smoke test. Arrays still retain the complete manifest length, with unprocessed rows marked by status `0`, so a later run with `EXTERNAL_MAX_ROWS=0` safely resumes the remainder. Set `EXTERNAL_RETRY_FAILED=1` to retry rows previously marked `-1`.


In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib
import json
import os
import sys
from fractions import Fraction

import numpy as np
from PIL import Image
from scipy.signal import resample_poly
import torch
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
import wfdb

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
CODE_ROOT = ARTIFACT_ROOT / "code"
module_path = CODE_ROOT / "ecg_cxr_ablation.py"
if not module_path.is_file():
    raise FileNotFoundError(f"Run notebook 12 first; missing {module_path}")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_ablation
importlib.reload(ecg_cxr_ablation)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
MAX_ROWS = int(os.environ.get("EXTERNAL_MAX_ROWS", "0"))
RETRY_FAILED = os.environ.get("EXTERNAL_RETRY_FAILED", "0").strip().lower() in {"1", "true", "yes"}
if MAX_ROWS < 0:
    raise ValueError("EXTERNAL_MAX_ROWS cannot be negative")

def read_manifest(modality):
    path = ARTIFACT_ROOT / "external" / modality / "manifest.jsonl"
    if not path.is_file():
        raise FileNotFoundError(f"Run notebook {'10' if modality == 'ecg' else '11'} first; missing {path}")
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not rows:
        raise ValueError(f"Empty {modality} manifest: {path}")
    required = ["sample_id", "split", "waveform_path" if modality == "ecg" else "image_path"]
    invalid = [(index, [key for key in required if key not in row or row[key] in (None, "")])
               for index, row in enumerate(rows)]
    invalid = [(index, missing) for index, missing in invalid if missing]
    if invalid:
        raise KeyError(f"{modality} manifest rows missing required fields: {invalid[:20]}")
    sample_ids = [str(row["sample_id"]) for row in rows]
    seen, duplicates = set(), set()
    for value in sample_ids:
        if value in seen:
            duplicates.add(value)
        seen.add(value)
    if duplicates:
        raise ValueError(f"Duplicate {modality} sample_id values: {sorted(duplicates)[:20]}")
    return rows

def manifest_sha256(rows):
    payload = json.dumps(rows, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()

manifests = {name: read_manifest(name) for name in ["ecg", "cxr"]}
manifest_hashes = {name: manifest_sha256(rows) for name, rows in manifests.items()}
print({name: {"rows": len(rows), "manifest_sha256": manifest_hashes[name][:12]}
       for name, rows in manifests.items()}, "device", DEVICE,
      "max_rows", MAX_ROWS, "retry_failed", RETRY_FAILED)


In [ ]:
LEADS = ["I", "II", "III", "AVR", "AVL", "AVF", "V1", "V2", "V3", "V4", "V5", "V6"]
PREPROCESSING_VERSION = "external_foundation_v2"

def load_ecg(stem):
    record = wfdb.rdrecord(str(stem), physical=True)
    signal = np.asarray(record.p_signal, dtype="float32")
    names = [str(value).upper().replace(" ", "") for value in record.sig_name]
    if not all(lead in names for lead in LEADS):
        raise ValueError(f"Missing canonical leads: {record.sig_name}")
    signal = signal[:, [names.index(lead) for lead in LEADS]]
    fs = float(record.fs)
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError(f"Invalid ECG sampling frequency: {record.fs}")
    if not np.isclose(fs, 500.0):
        ratio = (Fraction(500, 1) / Fraction(str(fs))).limit_denominator(1000)
        signal = resample_poly(signal, ratio.numerator, ratio.denominator, axis=0).astype("float32")
    if len(signal) < 5000:
        signal = np.pad(signal, ((0, 5000 - len(signal)), (0, 0)))
    signal = signal[:5000]
    mean, std = np.nanmean(signal, 0, keepdims=True), np.nanstd(signal, 0, keepdims=True)
    std[std < 1e-6] = 1
    return np.nan_to_num((signal - mean) / std).T.reshape(-1).astype("float32")

def load_cxr(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

def load_input(modality, row):
    return load_ecg(row["waveform_path"]) if modality == "ecg" else load_cxr(row["image_path"])

def load_foundation_encoder(modality):
    if modality == "ecg":
        model = AutoModel.from_pretrained(
            config["hubert_model_id"], cache_dir=config["hf_cache"],
            trust_remote_code=True, torch_dtype=DTYPE,
        ).to(DEVICE).eval().requires_grad_(False)
        return {"model": model}
    processor = AutoImageProcessor.from_pretrained(config["cxr_model_id"], cache_dir=config["hf_cache"])
    model = AutoModel.from_pretrained(
        config["cxr_model_id"], cache_dir=config["hf_cache"], torch_dtype=DTYPE,
    ).to(DEVICE).eval().requires_grad_(False)
    return {"model": model, "processor": processor}

@torch.inference_mode()
def encode_foundation(modality, inputs, bundle):
    if modality == "ecg":
        values = torch.from_numpy(np.stack(inputs)).to(DEVICE, dtype=DTYPE)
        output = bundle["model"](input_values=values)
        hidden = output.last_hidden_state if hasattr(output, "last_hidden_state") else output[0]
        pooled = hidden.mean(1) if hidden.ndim == 3 else hidden
    else:
        pixels = bundle["processor"](images=inputs, return_tensors="pt")["pixel_values"]
        output = bundle["model"](pixel_values=pixels.to(DEVICE, dtype=DTYPE))
        pooled = output.pooler_output if getattr(output, "pooler_output", None) is not None else output.last_hidden_state[:, 0]
    encoded = pooled.float().cpu().numpy()
    if encoded.ndim != 2 or len(encoded) != len(inputs) or not np.isfinite(encoded).all():
        raise ValueError(f"Invalid {modality} encoder output: shape={encoded.shape}")
    return encoded


In [ ]:
def read_metadata(path):
    try:
        value = json.loads(path.read_text())
        return value if isinstance(value, dict) else {}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def read_errors(path):
    try:
        return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def resumable_sensor_features(modality, rows, batch_size, bundle):
    if batch_size < 1:
        raise ValueError(f"{modality} batch size must be at least 1")
    out_dir = ARTIFACT_ROOT / "external" / modality
    feature_path = out_dir / f"{modality}_foundation.npy"
    status_path = out_dir / "encoding_status.npy"
    metadata_path = out_dir / "foundation_metadata.json"
    error_path = out_dir / "foundation_encoding_errors.jsonl"

    probe = None
    probe_errors = []
    for index, row in enumerate(rows):
        try:
            probe = encode_foundation(modality, [load_input(modality, row)], bundle)
            break
        except Exception as exc:
            probe_errors.append({"row_index": index, "sample_id": row["sample_id"],
                                 "stage": "probe", "error": repr(exc)})
    if probe is None:
        raise RuntimeError(f"No encodable {modality} record was found; examples={probe_errors[:10]}")

    shape = (len(rows), probe.shape[1])
    identity = {
        "modality": modality, "rows": len(rows),
        "manifest_sha256": manifest_hashes[modality],
        "encoder_model_id": config["hubert_model_id" if modality == "ecg" else "cxr_model_id"],
        "preprocessing_version": PREPROCESSING_VERSION, "output_dim": int(probe.shape[1]),
    }
    old_metadata = read_metadata(metadata_path)
    cache_matches = all(old_metadata.get(key) == value for key, value in identity.items())
    try:
        cache_matches = (cache_matches and feature_path.is_file() and status_path.is_file()
                         and np.load(feature_path, mmap_mode="r").shape == shape
                         and np.load(status_path, mmap_mode="r").shape == (len(rows),))
    except (OSError, ValueError):
        cache_matches = False
    if not cache_matches:
        features = np.lib.format.open_memmap(feature_path, mode="w+", dtype="float32", shape=shape)
        features[:] = np.nan; features.flush()
        status = np.lib.format.open_memmap(status_path, mode="w+", dtype="int8", shape=(len(rows),))
        status[:] = 0; status.flush()
        metadata_path.write_text(json.dumps({**identity, "complete": False}, indent=2))
    features = np.load(feature_path, mmap_mode="r+")
    status = np.load(status_path, mmap_mode="r+")
    previous_errors = read_errors(error_path)

    eligible = np.arange(min(MAX_ROWS, len(rows)) if MAX_ROWS > 0 else len(rows))
    pending = eligible[status[eligible] != 1] if RETRY_FAILED else eligible[status[eligible] == 0]
    errors = []

    def record_error(index, stage, exc):
        errors.append({"row_index": int(index), "sample_id": rows[int(index)]["sample_id"],
                       "stage": stage, "error": repr(exc)})

    for start in tqdm(range(0, len(pending), batch_size), desc=f"encode {modality}"):
        indices = pending[start:start + batch_size]
        inputs, valid = [], []
        for index in indices:
            try:
                inputs.append(load_input(modality, rows[index])); valid.append(int(index))
            except Exception as exc:
                status[index] = -1
                record_error(index, "load", exc)
        if inputs:
            try:
                encoded = encode_foundation(modality, inputs, bundle)
                features[valid] = encoded; status[valid] = 1
            except Exception:
                # Isolate a bad record instead of marking the entire batch failed.
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                for index, value in zip(valid, inputs):
                    try:
                        features[index] = encode_foundation(modality, [value], bundle)[0]
                        status[index] = 1
                    except Exception as exc:
                        status[index] = -1
                        record_error(index, "encode", exc)
        features.flush(); status.flush()

    error_by_index = {
        int(item["row_index"]): item for item in previous_errors
        if 0 <= int(item.get("row_index", -1)) < len(status)
        and status[int(item["row_index"])] == -1
    }
    for item in errors:
        error_by_index[int(item["row_index"])] = item
    current_errors = [error_by_index[index] for index in sorted(error_by_index)
                      if status[index] == -1]
    error_path.write_text("".join(json.dumps(item) + "\n" for item in current_errors))
    summary = {**identity, "selected_rows": len(eligible), "pending_processed": len(pending),
               "valid": int((status == 1).sum()), "failed": int((status == -1).sum()),
               "unprocessed": int((status == 0).sum()), "errors_this_run": len(errors),
               "complete": bool(np.all(status == 1))}
    metadata_path.write_text(json.dumps(summary, indent=2))
    print(modality, summary, feature_path)
    return summary

foundation_summaries = {}
for modality, batch_env, default_batch in [
    ("ecg", "EXTERNAL_ECG_BATCH_SIZE", 8),
    ("cxr", "EXTERNAL_CXR_BATCH_SIZE", 16),
]:
    bundle = load_foundation_encoder(modality)
    try:
        foundation_summaries[modality] = resumable_sensor_features(
            modality, manifests[modality], int(os.environ.get(batch_env, str(default_batch))), bundle)
    finally:
        del bundle
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
experiment_path = ARTIFACT_ROOT / "ablation" / "experiment_index.json"
if not experiment_path.is_file():
    raise FileNotFoundError(f"Run notebook 12 first; missing {experiment_path}")
experiment_index = json.loads(experiment_path.read_text())
if not isinstance(experiment_index, list) or not experiment_index:
    raise ValueError(f"No trained bridge experiments in {experiment_path}")
selected = os.environ.get("EXTERNAL_BRIDGE_VARIANTS", "")
if selected:
    requested = {value.strip() for value in selected.split(",") if value.strip()}
    available = {row["variant"] for row in experiment_index}
    unknown = requested - available
    if unknown:
        raise ValueError(f"Unknown EXTERNAL_BRIDGE_VARIANTS {sorted(unknown)}; available={sorted(available)}")
    experiment_index = [row for row in experiment_index if row["variant"] in requested]
BRIDGE_BATCH_SIZE = int(os.environ.get("EXTERNAL_BRIDGE_BATCH_SIZE", "1024"))
if BRIDGE_BATCH_SIZE < 1:
    raise ValueError("EXTERNAL_BRIDGE_BATCH_SIZE must be at least 1")

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

@torch.inference_mode()
def apply_bridge(row, modality, batch_size=BRIDGE_BATCH_SIZE):
    checkpoint_path = Path(row["checkpoint"])
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing bridge checkpoint: {checkpoint_path}")
    checkpoint = load_checkpoint(checkpoint_path)
    resolved = checkpoint["resolved_config"]
    model = ecg_cxr_ablation.AblationBridge(resolved["input_dims"], resolved["model_variant"],
                                            **resolved["model_hparams"]).to(DEVICE).eval().requires_grad_(False)
    model.load_state_dict(checkpoint["model"])
    if row["variant"] == "shuffled_pairs_control": model.variant = row["variant"]
    features = np.load(ARTIFACT_ROOT / "external" / modality / f"{modality}_foundation.npy", mmap_mode="r")
    status = np.load(ARTIFACT_ROOT / "external" / modality / "encoding_status.npy", mmap_mode="r")
    if len(features) != len(manifests[modality]) or len(status) != len(manifests[modality]):
        raise ValueError(f"{modality} foundation arrays do not match the manifest; rerun this notebook")
    expected_dim = int(resolved["input_dims"][modality])
    if features.ndim != 2 or features.shape[1] != expected_dim:
        raise ValueError(f"{modality} foundation dimension {features.shape} does not match bridge input {expected_dim}")
    out_dir = ARTIFACT_ROOT / "external" / modality / "bridge" / row["variant"] / f"seed_{row['seed']}"
    out_dir.mkdir(parents=True, exist_ok=True)
    common = np.lib.format.open_memmap(out_dir / "common.npy", mode="w+", dtype="float32", shape=(len(features), resolved["model_hparams"]["common_dim"]))
    continuous = np.lib.format.open_memmap(out_dir / "continuous_common.npy", mode="w+", dtype="float32", shape=common.shape)
    common[:] = np.nan; continuous[:] = np.nan
    indices_out = None
    if model.uses_vq:
        n_chunks = resolved["model_hparams"]["common_dim"] // resolved["model_hparams"]["code_dim"]
        indices_out = np.lib.format.open_memmap(out_dir / "common_indices.npy", mode="w+", dtype="int64", shape=(len(features), n_chunks))
        indices_out[:] = -1
    valid = np.where(status == 1)[0]
    if len(valid) == 0:
        raise RuntimeError(f"No valid external {modality} foundation features")
    for start in tqdm(range(0, len(valid), batch_size), desc=f"{row['variant']} {row['seed']} {modality}"):
        index = valid[start:start + batch_size]
        output = model.encode_modality(modality, torch.from_numpy(np.array(features[index])).to(DEVICE))
        common_batch = output.common.cpu().numpy()
        continuous_batch = output.continuous_common.cpu().numpy()
        if not np.isfinite(common_batch).all() or not np.isfinite(continuous_batch).all():
            raise ValueError(f"Non-finite bridge output for {row['variant']} seed {row['seed']} {modality}")
        common[index] = common_batch; continuous[index] = continuous_batch
        if indices_out is not None:
            index_batch = output.common_indices.cpu().numpy()
            codebook_size = int(resolved["model_hparams"]["common_codebook_size"])
            if index_batch.min() < 0 or index_batch.max() >= codebook_size:
                raise ValueError("Shared code index outside the configured codebook")
            indices_out[index] = index_batch
    common.flush(); continuous.flush()
    if indices_out is not None: indices_out.flush()
    metadata = {**row, "rows": len(features), "valid": len(valid),
                "manifest_sha256": manifest_hashes[modality],
                "foundation_model_id": config["hubert_model_id" if modality == "ecg" else "cxr_model_id"],
                "bridge_batch_size": batch_size}
    (out_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for experiment in experiment_index:
    for modality in ["ecg", "cxr"]:
        apply_bridge(experiment, modality)
print("External shared-code encoding complete.")
